# CSC786 Module 5 — NIDS Data Collection and Validation

This notebook documents a **reproducible data-acquisition and validation workflow** for the two primary University of Queensland NetFlow-v2 datasets used in the NIDS project. It does **not** train or evaluate detection models.

**Data access:** Download the original datasets from the official University of Queensland links in `README.md`. Do not add the original CSVs to a public GitHub repository. Place the extracted primary CSV files into `data/raw/` after cloning the repository in Colab. The dataset files may be large; use Google Drive or another permitted storage location if Colab's temporary disk is insufficient. Only CSVs present in `data/raw/` will be processed, and missing datasets will be reported.

**Data protection:** The code does not publish raw records. Local sampled outputs omit common IP-address and flow-identifier fields and remain Git-ignored. This is a data-preparation demonstration, not a claim that the resulting sample is representative of all attack categories.

**Official dataset information:** https://staff.itee.uq.edu.au/marius/NIDS_datasets/  
**Dataset citation:** Sarhan, Layeghy, and Portmann, *Towards a Standard Feature Set for Network Intrusion Detection System Datasets*, https://doi.org/10.1007/s11036-021-01843-0

## 1. Setup and fixed parameters

In a **separate Colab notebook**, clone your repository and change into its folder:

```python
!git clone https://github.com/anjanarprasad/csc786-ethical-data-workflow.git
%cd csc786-ethical-data-workflow
!pip install -r requirements.txt
```

Then open this notebook from the cloned repository and run all cells. If you start this notebook directly from Colab's GitHub tab, ensure the runtime's working directory is the cloned repository folder. For a repeatable comparison, use identical source files, parameters, and software versions. Dates below indicate *processing time*, not download time.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import sys

import numpy as np
import pandas as pd

REPO_NAME = "csc786-ethical-data-workflow"
SOURCES = {
    "NF-CSE-CIC-IDS2018-v2": "https://rdm.uq.edu.au/files/ce5161d0-ef9c-11ed-827d-e762de186848",
    "NF-UNSW-NB15-v2": "https://rdm.uq.edu.au/files/8c6e2a00-ef9c-11ed-827d-e762de186848",
}
SEED = 42
CHUNK_SIZE = 50_000
SAMPLE_PER_CLASS = 2_000  # Max 2,000 benign and 2,000 malicious flows per dataset
# Enter actual download dates in YYYY-MM-DD format after retrieving the files.
# Leave None rather than inventing a date.
DOWNLOAD_DATES = {name: None for name in SOURCES}

candidates = [Path.cwd(), Path.cwd() / REPO_NAME, Path.cwd().parent]
ROOT = next((path for path in candidates if (path / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Please clone the repository and %cd into it before running this notebook.")
RAW_DIR = ROOT / "data" / "raw"
OUTPUT_DIR = ROOT / "data" / "processed"
PROVENANCE_DIR = ROOT / "data" / "provenance"
for folder in (RAW_DIR, OUTPUT_DIR, PROVENANCE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT.resolve())
print("Raw data directory:", RAW_DIR.resolve())
print("Python:", platform.python_version(), "pandas:", pd.__version__, "numpy:", np.__version__)
print("Seed:", SEED, "Chunk size:", CHUNK_SIZE, "Max sample per binary class:", SAMPLE_PER_CLASS)

## 2. Locate downloaded CSV files

The notebook looks for CSV filenames containing the selected dataset name (case-insensitive). It does not silently substitute an unrelated dataset. If no valid file is present, it stops and tells you which files are missing.

In [ ]:
def normalize_filename(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

csv_files = sorted(RAW_DIR.rglob("*.csv"))
matched = {}
missing = []
for name in SOURCES:
    matches = [path for path in csv_files
               if normalize_filename(name) in normalize_filename(path.stem)]
    if len(matches) > 1:
        raise ValueError(f"Multiple CSVs match {name}: {matches}. Keep one original CSV for each dataset.")
    if matches:
        matched[name] = matches[0]
    else:
        missing.append(name)

print("Available primary datasets:")
for name, path in matched.items():
    print(f"  {name}: {path.relative_to(ROOT)}")
if missing:
    print("Not available in data/raw/:", ", ".join(missing))
if not matched:
    raise FileNotFoundError(
        "No primary dataset CSVs found. Download at least one v2 CSV using the links in README.md, "
        "place it under data/raw/, and rerun the notebook."
    )

## 3. Stream validation and seeded sampling

The code reads the dataset in chunks to avoid loading all records into RAM. It checks the schema and binary labels, counts missing values and class proportions across the **entire input file**, and produces a deterministic, per-class sample using seeded random priorities. Duplicate counts are checked only within the final sample (not the full file). Original CSVs are never modified.

If rows have unexpected labels, validation fails rather than assigning them a class. **No train/test splitting or model fitting happens here.**

In [ ]:
SENSITIVE_COLUMNS = {
    "IPV4_SRC_ADDR", "IPV4_DST_ADDR", "IPV6_SRC_ADDR", "IPV6_DST_ADDR",
    "SRC_IP", "DST_IP", "SOURCE_IP", "DESTINATION_IP", "FLOW_ID",
}
LABEL_VALUES = {"0": 0, "1": 1, "0.0": 0, "1.0": 1,
                "benign": 0, "normal": 0, "malicious": 1, "attack": 1}

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

def binary_labels(values):
    normalized = values.astype("string").str.strip().str.lower()
    return normalized.map(LABEL_VALUES)

def validate_and_sample(dataset_name, csv_path, seed):
    rng = np.random.default_rng(seed)
    samples = {0: pd.DataFrame(), 1: pd.DataFrame()}
    class_counts = {0: 0, 1: 0}
    missing_by_column = None
    original_columns = None
    total_rows = 0
    invalid_labels = 0
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE, low_memory=False):
        chunk.columns = [str(name).strip() for name in chunk.columns]
        if original_columns is None:
            original_columns = list(chunk.columns)
            label_matches = [name for name in chunk.columns if name.lower() == "label"]
            if len(label_matches) != 1:
                raise ValueError(f"{dataset_name}: expected one binary 'Label' column, got {label_matches}")
            label_column = label_matches[0]
            missing_by_column = pd.Series(0, index=original_columns, dtype="int64")
        elif list(chunk.columns) != original_columns:
            raise ValueError(f"{dataset_name}: inconsistent CSV columns")
        total_rows += len(chunk)
        missing_by_column = missing_by_column.add(chunk.isna().sum(), fill_value=0).astype("int64")
        labels = binary_labels(chunk[label_column])
        invalid_labels += int(labels.isna().sum())
        priorities = rng.random(len(chunk))
        for label in (0, 1):
            row_mask = labels.eq(label).fillna(False).to_numpy(dtype=bool)
            class_counts[label] += int(row_mask.sum())
            if not row_mask.any():
                continue
            frame = chunk.loc[row_mask].copy()
            frame["binary_label"] = label
            frame["__sample_priority"] = priorities[row_mask]
            combined = pd.concat([samples[label], frame], ignore_index=True)
            samples[label] = combined.nsmallest(SAMPLE_PER_CLASS, "__sample_priority")
    if total_rows == 0:
        raise ValueError(f"{dataset_name}: empty CSV")
    if invalid_labels:
        raise ValueError(f"{dataset_name}: {invalid_labels} labels not recognized as binary 0/1")
    selected = pd.concat([samples[0], samples[1]], ignore_index=True)
    selected = selected.sort_values("__sample_priority").drop(columns=["__sample_priority"])
    selected["source_dataset"] = dataset_name
    removed = [c for c in selected.columns if c.upper().replace(" ", "_") in SENSITIVE_COLUMNS]
    selected = selected.drop(columns=removed)
    selected = selected.reset_index(drop=True)
    info = {
        "records_in_original_csv": int(total_rows),
        "original_column_count": len(original_columns),
        "columns_in_original_csv": original_columns,
        "class_counts_full_data": {str(k): v for k, v in class_counts.items()},
        "missing_values_full_data": {col: int(count) for col, count in missing_by_column.items() if count > 0},
        "sample_rows": int(len(selected)),
        "sample_class_counts": {str(k): int((selected['binary_label'] == k).sum()) for k in (0, 1)},
        "duplicate_rows_in_sample": int(selected.duplicated().sum()),
        "excluded_columns_in_sample": removed,
    }
    return selected, info

## 4. Save outputs and collection provenance

Each run records its UTC processing timestamp, manually supplied download date (if known), original file SHA-256 hash, software versions, parameters, validation summary, and the sampled CSV hash. Generated files are intentionally kept under `data/` and are ignored by Git. Use the printed run information to update the **Collection Runs** table in `DATA_README.md`.

**Note:** Exact sample hashes are meaningful only when the source CSV, row ordering, parameters, and software serialization behavior are unchanged.

In [ ]:
run_summaries = []
for name, file_path in matched.items():
    dataset_seed = SEED + list(SOURCES).index(name)
    print("\nProcessing:", name)
    sample, validation = validate_and_sample(name, file_path, seed=dataset_seed)
    source_sha256 = sha256_file(file_path)
    stem = name.lower()
    output_csv = OUTPUT_DIR / f"{stem}_sample.csv"
    sample.to_csv(output_csv, index=False)
    generated_sha256 = sha256_file(output_csv)
    run = {
        "dataset": name,
        "official_source": SOURCES[name],
        "relative_input_path": str(file_path.relative_to(ROOT)),
        "download_date_utc_user_recorded": DOWNLOAD_DATES[name],
        "processing_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "source_size_bytes": int(file_path.stat().st_size),
        "source_sha256": source_sha256,
        "output_relative_path": str(output_csv.relative_to(ROOT)),
        "output_sha256": generated_sha256,
        "parameters": {"random_seed": dataset_seed, "chunk_size": CHUNK_SIZE,
                       "max_sample_per_class": SAMPLE_PER_CLASS},
        "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                        "numpy": np.__version__},
        "validation": validation,
        "status": "completed",
    }
    manifest = PROVENANCE_DIR / f"{stem}_provenance.json"
    manifest.write_text(json.dumps(run, indent=2), encoding="utf-8")
    run_summaries.append({
        "dataset": name,
        "processed_utc": run["processing_timestamp_utc"],
        "download_date_utc": run["download_date_utc_user_recorded"],
        "records_loaded": validation["records_in_original_csv"],
        "sample_rows": validation["sample_rows"],
        "source_sha256": source_sha256,
        "sample_sha256": generated_sha256,
        "status": "completed",
    })
    print("Records:", validation["records_in_original_csv"],
          "Binary classes:", validation["class_counts_full_data"])
    print("Missing-value columns:", len(validation["missing_values_full_data"]),
          "Sample duplicate rows:", validation["duplicate_rows_in_sample"])
    print("Saved sample:", output_csv.relative_to(ROOT))
    print("Source SHA-256:", source_sha256)
    print("Sample SHA-256:", generated_sha256)

run_log = ROOT / "data" / "collection_runs.csv"
new_runs = pd.DataFrame(run_summaries)
if run_log.exists():
    previous = pd.read_csv(run_log)
    new_runs = pd.concat([previous, new_runs], ignore_index=True)
new_runs.to_csv(run_log, index=False)
print("\nSaved collection log:", run_log.relative_to(ROOT))
print("Processed datasets:", len(run_summaries), "of", len(SOURCES))
print("Missing datasets:", missing)
display(pd.DataFrame(run_summaries)[["dataset", "records_loaded", "sample_rows", "status"]])

## 5. Check outputs and report limitations

Verify sample files and manifests under `data/`. Compare saved SHA-256 values on a repeat run. If the original file bytes or package versions differ, investigate before assuming exact reproducibility. If only one primary dataset is present, the run demonstrates partial collection, **not** full cross-dataset preparation. The sample is for validation and reproducibility; do not use it for unbiased model performance claims without a separate experimental design.

In [ ]:
for item in run_summaries:
    sample_path = OUTPUT_DIR / f"{item['dataset'].lower()}_sample.csv"
    sample_df = pd.read_csv(sample_path)
    assert "binary_label" in sample_df.columns
    assert set(sample_df["binary_label"].dropna().unique()).issubset({0, 1})
    assert len(sample_df) <= 2 * SAMPLE_PER_CLASS
    assert sha256_file(sample_path) == item["sample_sha256"]
    assert not any(c.upper().replace(" ", "_") in SENSITIVE_COLUMNS for c in sample_df.columns)
    print(f"Verified {item['dataset']}: {len(sample_df)} sampled rows; file checksum matches")
print("Validation finished. Do not commit the files inside data/raw, data/processed, or data/provenance.")